In [1]:
# mount to drive
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive", force_remount=False)

if Path("/content/drive/MyDrive/training-embedding").exists():
    project_root = "/content/drive/MyDrive/training-embedding"
elif Path("/content/drive/My Drive/training-embedding").exists():
    project_root = "/content/drive/My Drive/training-embedding"
else:
    raise RuntimeError("training-embedding folder not found under mounted Drive.")

models_root = Path(project_root) / "models"
print(f"project_root={project_root}")
print(f"models_root={models_root}")


Mounted at /content/drive
project_root=/content/drive/MyDrive/training-embedding
models_root=/content/drive/MyDrive/training-embedding/models


In [2]:
# install bitsandbytes
%%bash
set -euo pipefail

python -m pip install -U pip
python -m pip install -U "bitsandbytes>=0.46.1" accelerate transformers peft


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 39.3 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 40.0 MB/s  0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 106.0 MB/s  0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0



In [3]:
# detect models
from pathlib import Path

candidates = sorted([p for p in models_root.iterdir() if p.is_dir()]) if models_root.exists() else []
if not candidates:
    raise RuntimeError(f"No model folders found under: {models_root}")

print("Available model folders:")
for idx, path in enumerate(candidates):
    print(f"[{idx}] {path}")


Available model folders:
[0] /content/drive/MyDrive/training-embedding/models/gguf_cache
[1] /content/drive/MyDrive/training-embedding/models/turkish-gemma-9b-t1-4bit


In [4]:
# load model
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

preferred_dirs = [
    models_root / 'turkish-gemma-9b-t1-4bit',
    models_root / 'turkish-gemma-9b-v01-4bit',
]

MODEL_DIR = next((str(p) for p in preferred_dirs if p.exists()), None)
if MODEL_DIR is None:
    raise RuntimeError(
        f'Could not find expected quantized model folder. Checked: {[str(p) for p in preferred_dirs]}'
    )

tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    device_map='auto',
    dtype=torch.float16,
)

print('Loaded model from:', MODEL_DIR)
print('Model device:', model.device)


The following generation flags are not valid and may be ignored: ['cache_implementation']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/464 [00:00<?, ?it/s]

Loaded model from: /content/drive/MyDrive/training-embedding/models/turkish-gemma-9b-t1-4bit
Model device: cuda:0


In [6]:
# model inference
PROMPT = "Senin adın nedir?"

inputs = tokenizer(PROMPT, return_tensors="pt")
inputs = {k: v.to(model.device) for k, v in inputs.items()}

with torch.inference_mode():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=4096,
        do_sample=False,
    )

text = tokenizer.decode(output_ids[0], skip_special_tokens=True)
print(text)


KeyboardInterrupt: 